# 01 - Data Cleaning & EDA : Online Retail II
Goal: load the row transactions, inspect data quality issue, document every cleaning decision with row counts, and save a clean dataset for RFM, market basket and recomendation notebooks.

In [1]:
import pandas as pd
import numpy as np
from pathlib  import Path

pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_XLSX = RAW_DIR / "online_retail_II.xlsx"
RAW_PARQUET = RAW_DIR / "online_retail_raw.parquet"

print(PROJECT_DIR)
print("Excel found:", RAW_XLSX.exists())

C:\Users\user\Desktop\market-basket-next-product
Excel found: True


In [2]:
%%time
sheets = pd.read_excel(RAW_XLSX, sheet_name=None) # dict: {sheet name: DataFrame}

frames = []
for name, sheet_df in sheets.items():
    print(f"{name}: {sheet_df.shape}")
    frames.append(sheet_df.assign(sheet=name))   # add column with sheet name

df = pd.concat(frames, ignore_index = True)    # stack sheets vertically
print("Combined:", df.shape)
df.head()

Year 2009-2010: (525461, 8)
Year 2010-2011: (541910, 8)
Combined: (1067371, 9)
CPU times: total: 3min 1s
Wall time: 3min 1s


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [3]:
df = df.astype({
    "Invoice":     "string",
    "StockCode":   "string",
    "Description": "string",
    "Country":     "string",
    "sheet":       "string",
    "Customer ID": "Int64",
})
print(df.dtypes)

df.to_parquet(RAW_PARQUET, index=False)
print("Saved:", RAW_PARQUET.name, f"{RAW_PARQUET.stat().st_size / 1e6:.1f} MB")

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                string
sheet                  string
dtype: object
Saved: online_retail_raw.parquet 7.2 MB


In [4]:
%%time
df = pd.read_parquet(RAW_PARQUET)
print(df.shape)
df.dtypes

(1067371, 9)
CPU times: total: 625 ms
Wall time: 613 ms


Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                string
sheet                  string
dtype: object

In [5]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct":     (df.isna().mean() * 100).round(2),
})
missing

,missing,pct
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00
sheet,0,0.00


In [6]:
print(df.groupby("sheet")["InvoiceDate"].agg(["min", "max", "count"]))

start_2nd = df.loc[df["sheet"] == "Year 2010-2011", "InvoiceDate"].min()
overlap = df[(df["sheet"] == "Year 2009-2010") & (df["InvoiceDate"] >= start_2nd)]
print("\nRows in sheet 1 dated after sheet 2 starts:", len(overlap))

                               min                 max   count
sheet                                                         
Year 2009-2010 2009-12-01 07:45:00 2010-12-09 20:01:00  525461
Year 2010-2011 2010-12-01 08:26:00 2011-12-09 12:50:00  541910

Rows in sheet 1 dated after sheet 2 starts: 22523


In [7]:
print("Full duplicates (incl. sheet column):", df.duplicated().sum())
print("Duplicates ignoring sheet column:   ", df.drop(columns="sheet").duplicated().sum())

Full duplicates (incl. sheet column): 12133
Duplicates ignoring sheet column:    34335


In [8]:
in_overlap = (df["sheet"] == "Year 2009-2010") & (df["InvoiceDate"] >= start_2nd)
df_no_overlap = df[~in_overlap]

print("Rows after removing overlap:", len(df_no_overlap))
print("Duplicates left:", df_no_overlap.drop(columns="sheet").duplicated().sum())

Rows after removing overlap: 1044848
Duplicates left: 11812


In [9]:
is_cancel = df["Invoice"].str.startswith("C")
neg_qty   = df["Quantity"] < 0
bad_price = df["Price"] <= 0

print("Cancellation invoices (C...):", is_cancel.sum())
print("Quantity < 0:               ", neg_qty.sum())
print("Price <= 0:                 ", bad_price.sum())

print("\nCancellation vs negative quantity:")
print(pd.crosstab(is_cancel, neg_qty, rownames=["cancel"], colnames=["qty<0"]))

print("\nInvoice prefixes that are not plain numbers:")
print(df.loc[~df["Invoice"].str.isdigit(), "Invoice"].str[0].value_counts())

print("\nShare of Price<=0 rows with missing Description:",
      round(df.loc[bad_price, "Description"].isna().mean(), 3))

Cancellation invoices (C...): 19494
Quantity < 0:                22950
Price <= 0:                  6207

Cancellation vs negative quantity:
qty<0     False  True 
cancel                
False   1044420   3457
True          1  19493

Invoice prefixes that are not plain numbers:
Invoice
C    19494
A        6
Name: count, dtype: int64[pyarrow]

Share of Price<=0 rows with missing Description: 0.706


In [10]:
df[neg_qty & ~is_cancel].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,sheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,Year 2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom,Year 2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,<NA>,United Kingdom,Year 2009-2010
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom,Year 2009-2010


In [11]:
is_std = df["StockCode"].str.match(r"^\d{5}[A-Za-z]*$")
print("Rows with non-standard StockCode:", (~is_std).sum(),
      "| unique codes:", df.loc[~is_std, "StockCode"].nunique())

(df[~is_std]
 .groupby("StockCode")
 .agg(rows=("Invoice", "size"),
      example=("Description", "first"),
      avg_price=("Price", "mean"))
 .sort_values("rows", ascending=False)
 .head(25))

Rows with non-standard StockCode: 6094 | unique codes: 63


,rows,example,avg_price
StockCode,,,
POST,2122,POSTAGE,32.623322
DOT,1446,DOTCOM POSTAGE,223.144876
M,1421,Manual,528.776284
C2,282,CARRIAGE,49.460993
D,177,Discount,72.867288
S,104,SAMPLES,60.956731
BANK CHARGES,102,Bank Charges,359.034422
ADJUST,67,Adjustment by john on 26/01/2010 16,163.591343
AMAZONFEE,43,AMAZON FEE,7646.089302


In [12]:
print("Unique StockCodes:  ", df["StockCode"].nunique())
print("Unique Descriptions:", df["Description"].nunique())

desc_per_code = df.groupby("StockCode")["Description"].nunique()
print("Codes with >1 description:", (desc_per_code > 1).sum())

worst = desc_per_code.idxmax()
print(f"\nExample — {worst}:")
print(df.loc[df["StockCode"] == worst, "Description"].value_counts())

Unique StockCodes:   5305
Unique Descriptions: 5698
Codes with >1 description: 1232

Example — 20713:
Description
JUMBO BAG OWLS                  1372
missing                            1
wrongly marked. 23343 in box       1
wrongly coded-23343                1
found                              1
Found                              1
wrongly marked 23343               1
Marked as 23343                    1
wrongly coded 23343                1
Name: count, dtype: int64[pyarrow]


In [13]:
has_lower = df["Description"].str.contains(r"[a-z]", na=False)
print("Descriptions with lowercase letters:", has_lower.sum())
df.loc[has_lower, "Description"].value_counts().head(30)

Descriptions with lowercase letters: 5967


Description
Manual                                 1426
BAG 250g SWIRLY MARBLES                 598
BAG 125g SWIRLY MARBLES                 596
POLYESTER FILLER PAD 40x40cm            398
POLYESTER FILLER PAD 45x45cm            372
3 TRADITIONAl BISCUIT CUTTERS  SET      268
BAG 500g SWIRLY MARBLES                 240
FRENCH BLUE METAL DOOR SIGN No          186
Discount                                177
check                                   162
Bank Charges                             96
damages                                  84
damaged                                  81
ESSENTIAL BALM 3.5g TIN IN ENVELOPE      80
Next Day Carriage                        80
POLYESTER FILLER PAD 45x30cm             65
POLYESTER FILLER PAD 65CMx65CM           52
POLYESTER FILLER PAD 30CMx30CM           52
FRENCH BLUE METAL DOOR SIGN, No          48
NUMBER TILE COTTAGE GARDEN No            42
NUMBER TILE VINTAGE FONT No              39
Adjustment by john on 26/01/2010 16      38
FOLK ART GREETING CA

In [17]:
(df[~is_std]
 .groupby("StockCode")
 .agg(rows=("Invoice", "size"),
      example=("Description", "first"),
      avg_price=("Price", "mean"))
 .sort_values("rows", ascending=False)
 .iloc[25:])

,rows,example,avg_price
StockCode,,,
DCGS0004,5,HAYNES CAMPER SHOULDER BAG,13.588000
DCGS0066N,4,NAVY CUDDLES DOG HOODIE,4.325000
DCGS0072,4,CAT CAMOUFLAGUE COLLAR,3.877500
gift_0001_80,4,Dotcomgiftshop Gift Voucher £80.00,34.780000
ADJUST2,3,Adjustment by Peter on Jun 25 2010,243.683333
DCGS0070,3,CAMOUFLAGE DOG COLLAR,8.480000
gift_0001_70,3,Dotcomgiftshop Gift Voucher £70.00,19.856667
DCGS0068,3,DOGS NIGHT COLLAR,5.706667
SP1002,3,KID'S CHALKBOARD/EASEL,1.966667


In [20]:
print(df["StockCode"].nunique(), "vs uppercased:", df["StockCode"].str.upper().nunique())

5305 vs uppercased: 5132


In [31]:
NON_PRODUCT_CODES = {"POST", "DOT", "M", "C2", "C3", "D", "S",
                     "BANK CHARGES", "AMAZONFEE", "CRUK", "B", "23444"}
NON_PRODUCT_PATTERN = r"^(ADJUST|TEST|GIFT)"      # ADJUST2, TEST002, GIFT_0001_30...

clean = df.copy()
log = [{"step": "0. Raw data (both sheets)", "rows_removed": 0, "rows_left": len(clean)}]

def drop(data, mask, step):
    """Remove rows where mask is True and record how many were removed."""
    removed = int(mask.sum())
    data = data[~mask]
    log.append({"step": step, "rows_removed": removed, "rows_left": len(data)})
    return data

# 1. Overlap: take Dec 1–9 2010 from sheet 2 only
clean = drop(clean, (clean["sheet"] == "Year 2009-2010") & (clean["InvoiceDate"] >= start_2nd),
             "1. Sheet overlap (Dec 2010 duplicated in both sheets)")

# Normalise codes before checking duplicates (85123a = 85123A)
clean["StockCode"] = clean["StockCode"].str.strip().str.upper()

clean = drop(clean, clean.drop(columns="sheet").duplicated(),      "2. Exact duplicate rows")
clean = drop(clean, clean["Invoice"].str.startswith("C"),           "3. Cancellations (C invoices)")
clean = drop(clean, clean["Invoice"].str.startswith("A"),           "4. Bad-debt adjustments (A invoices)")
clean = drop(clean, clean["Quantity"] <= 0,                         "5. Quantity <= 0 (stock adjustments)")
clean = drop(clean, clean["Price"] <= 0,                            "6. Price <= 0")
clean = drop(clean, clean["StockCode"].isin(NON_PRODUCT_CODES)
                    | clean["StockCode"].str.match(NON_PRODUCT_PATTERN),
             "7. Non-product codes (postage, fees, vouchers, tests)")

clean = clean.drop(columns="sheet").reset_index(drop=True)

cleaning_log = pd.DataFrame(log)
cleaning_log

,step,rows_removed,rows_left
0,0. Raw data (both sheets),0,1067371
1,1. Sheet overlap (Dec 2010 duplicated in both ...,22523,1044848
2,2. Exact duplicate rows,11812,1033036
3,3. Cancellations (C invoices),19104,1013932
4,4. Bad-debt adjustments (A invoices),6,1013926
5,5. Quantity <= 0 (stock adjustments),3393,1010533
6,6. Price <= 0,2621,1007912
7,"7. Non-product codes (postage, fees, vouchers,...",4634,1003278


In [32]:
print("Missing values:\n", clean.isna().sum(), "\n")
print("Customer ID missing share:", round(clean["Customer ID"].isna().mean(), 3))
print("Unique StockCodes:", clean["StockCode"].nunique())

desc_per_code = clean.groupby("StockCode")["Description"].nunique()
print("Codes with >1 description:", (desc_per_code > 1).sum())

has_lower = clean["Description"].str.contains(r"[a-z]", na=False)
clean.loc[has_lower, "Description"].value_counts().head(20)

Missing values:
 Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    226761
Country             0
dtype: int64 

Customer ID missing share: 0.226
Unique StockCodes: 4724
Codes with >1 description: 624


Description
BAG 125g SWIRLY MARBLES                577
BAG 250g SWIRLY MARBLES                574
POLYESTER FILLER PAD 40x40cm           380
POLYESTER FILLER PAD 45x45cm           354
3 TRADITIONAl BISCUIT CUTTERS  SET     252
BAG 500g SWIRLY MARBLES                228
FRENCH BLUE METAL DOOR SIGN No         173
ESSENTIAL BALM 3.5g TIN IN ENVELOPE     74
POLYESTER FILLER PAD 45x30cm            61
POLYESTER FILLER PAD 30CMx30CM          51
POLYESTER FILLER PAD 65CMx65CM          49
FRENCH BLUE METAL DOOR SIGN, No         48
NUMBER TILE COTTAGE GARDEN No           40
NUMBER TILE VINTAGE FONT No             36
FOLK ART GREETING CARD,pack/12          28
THE KING GIFT BAG 25x24x12cm            13
Hypochondriac Pill Stress Ball           9
Dad's Cab Electronic Meter               9
POLYESTER FILLER PAD 60x40cm             6
Dr. Jam's Arouzer Stress Ball            5
Name: count, dtype: int64[pyarrow]

In [33]:
clean.loc[clean["Description"] == "Next Day Carriage", ["StockCode", "Price"]].drop_duplicates()

,StockCode,Price


In [34]:
multi = desc_per_code[desc_per_code > 1].sort_values(ascending=False)
for code in multi.index[:5]:
    print(code)
    print(clean.loc[clean["StockCode"] == code, "Description"].value_counts(), "\n")

22344
Description
PARTY PIZZA DISH PINK POLKADOT       88
PARTY PIZZA DISH PINK RETROSPOT      24
PARTY PIZZA DISH PINK WHITE SPOT     19
PARTY PIZZA DISH PINK+WHITE SPOT      2
Name: count, dtype: int64[pyarrow] 

20685
Description
DOORMAT RED RETROSPOT      901
DOOR MAT RED SPOT          412
RED SPOTTY COIR DOORMAT    208
DOORMAT RED SPOT           207
Name: count, dtype: int64[pyarrow] 

21955
Description
DOORMAT UNION JACK GUNS AND ROSES      344
DOOR MAT UNION JACK GUNS AND ROSES     228
UNION JACK GUNS & ROSES  DOORMAT       102
  DOORMAT UNION JACK GUNS AND ROSES     60
Name: count, dtype: int64[pyarrow] 

22345
Description
PARTY PIZZA DISH BLUE POLKADOT       83
PARTY PIZZA DISH BLUE RETROSPOT      28
PARTY PIZZA DISH BLUE WHITE SPOT     17
PARTY PIZZA DISH BLUE+WHITE SPOT      2
Name: count, dtype: int64[pyarrow] 

22346
Description
PARTY PIZZA DISH GREEN POLKADOT       69
PARTY PIZZA DISH GREEN RETROSPOT      19
PARTY PIZZA DISH GREEN WHITE SPOT     15
PARTY PIZZA DISH GREEN+

In [35]:
canonical = (clean.groupby("StockCode")["Description"]
                  .agg(lambda s: s.value_counts().index[0]))   # most frequent name
clean["Description"] = clean["StockCode"].map(canonical)

print("Codes with >1 description:", clean.groupby("StockCode")["Description"].nunique().gt(1).sum())

Codes with >1 description: 0


In [36]:
df.loc[df["StockCode"].str.upper() == "23444", "Description"].value_counts()

Description
Next Day Carriage    80
Name: count, dtype: int64[pyarrow]

In [37]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
clean.to_parquet(PROCESSED_DIR / "online_retail_clean.parquet", index=False)
cleaning_log.to_csv(PROCESSED_DIR / "cleaning_log.csv", index=False)

print("Rows:      ", f"{len(clean):,}")
print("Invoices:  ", f"{clean['Invoice'].nunique():,}")
print("Customers: ", f"{clean['Customer ID'].nunique():,}")
print("Products:  ", f"{clean['StockCode'].nunique():,}")
print("Countries: ", clean["Country"].nunique())
print("Period:    ", clean["InvoiceDate"].min(), "→", clean["InvoiceDate"].max())

Rows:       1,003,278
Invoices:   39,516
Customers:  5,852
Products:   4,724
Countries:  43
Period:     2009-12-01 07:45:00 → 2011-12-09 12:50:00
